Nama Kelompok :

24083010044 Via Amanda

24083010057 Maulida Aprilia P.H.

24083010059 Izzati Kamila P.

In [ ]:
!pip install -q translators langdetect symspellpy sastrawi

In [ ]:
# Import pustaka umum
import pandas as pd
import re
import string
import unicodedata

# Untuk translasi otomatis
import translators as ts

# Deteksi bahasa
from langdetect import detect, LangDetectException
from langdetect import DetectorFactory

# Koreksi ejaan
from symspellpy import SymSpell, Verbosity

# Tokenisasi dan NLP dasar
import spacy
import nltk
from nltk.tokenize import word_tokenize

# Model NLP berbasis transformer (jika dibutuhkan)
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# Utilitas
from collections import Counter

# Pustaka Bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
df = pd.read_csv('/content/BIG DATA BPJS PBI - Gabungan.csv')
df

,Date,Username,Comment
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...
2,12/2/2025,grok,@AntonSubagia @Bank_Joee_ @susno2g @Jediimar @...
3,12/3/2025,yourrsse,@BPJSKesehatanRI min kalo mau pindah dari bpjs...
4,12/3/2025,hamiyaaaa,Bpjs pbi akhir bln kemarin serentak pada nonak...
...,...,...,...
576,3/2/2026,subisaja,@tanyarlfes BPJS @BPJSKesehatanRI PBI gimana i...
577,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi BPJS PBI yang diblokir...
578,3/2/2026,perju_angan,@jam1malamm @realDonaldTrump He don't understa...
579,3/2/2026,maysa99628,"@KemensosRI Kemensos kerja nyata, semoga denga..."


In [ ]:
df = df.rename(columns={'Comment': 'text'})
df

,Date,Username,text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...
2,12/2/2025,grok,@AntonSubagia @Bank_Joee_ @susno2g @Jediimar @...
3,12/3/2025,yourrsse,@BPJSKesehatanRI min kalo mau pindah dari bpjs...
4,12/3/2025,hamiyaaaa,Bpjs pbi akhir bln kemarin serentak pada nonak...
...,...,...,...
576,3/2/2026,subisaja,@tanyarlfes BPJS @BPJSKesehatanRI PBI gimana i...
577,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi BPJS PBI yang diblokir...
578,3/2/2026,perju_angan,@jam1malamm @realDonaldTrump He don't understa...
579,3/2/2026,maysa99628,"@KemensosRI Kemensos kerja nyata, semoga denga..."


In [ ]:
df['text'] = df['text'].str.lower()
df

,Date,Username,text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...
...,...,...,...
576,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...
577,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...
578,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...
579,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga..."


In [ ]:
!wget -O colloquial-dict.csv https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv

colloquial_csv = pd.read_csv('colloquial-dict.csv', usecols = ['slang','formal'])
colloquial_csv.head(10)

colloquial_dict = dict(colloquial_csv.values)

def replace_colloquial(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'(\d+)(\D)', r'\1 \2', text)
    return ' '.join([colloquial_dict.get(word, word) for word in text.split()])

df['colluqial_transformed_text'] = df['text'].apply(replace_colloquial)
df

--2026-03-16 16:31:50--  https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3093160 (2.9M) [text/plain]
Saving to: ‘colloquial-dict.csv’

colloquial-dict.csv 100%[===================>]   2.95M  --.-KB/s    in 0.02s   

2026-03-16 16:31:50 (133 MB/s) - ‘colloquial-dict.csv’ saved [3093160/3093160]



,Date,Username,text,colluqial_transformed_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,@undipmfs2 waktu itu aku case-nya juga sama ka...
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,@antonsubagia @bank_joee_ @susno2 enggak @jedi...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,@bpjskesehatanri min kalo mau pindah dari bpjs...
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...
...,...,...,...,...
576,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,@tanyarlfes bpjs @bpjskesehatanri pbi bagaiman...
577,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,@heywannaknoww @nyunssi bpjs pbi yang diblokir...
578,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,@jam1 malamm @realdonaldtrump he don't underst...
579,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga...","@kemensosri kemensos kerja nyata, semoga denga..."


In [ ]:
import re

df['colluqial_transformed_text'] = df['colluqial_transformed_text'].str.replace(r'@\w+', '', regex=True)

In [ ]:
def clean_text(text):
    marker = "SPACEMARKER"

    char_list = []
    for char in text:
        if ord(char) < 128:
            char_list.append(char)
        else:
            char_list.append(marker)

    marked_text = ''.join(char_list)

    cleaned_text = marked_text.replace(marker, ' ')

    cleaned_text = re.sub(r'\d+', '', cleaned_text)

    cleaned_text = re.sub(r'[^\w\s]', ' ', cleaned_text)

    words = cleaned_text.split()
    cleaned_words = [re.sub(r'(.)\1{2,}', r'\1\1', word) for word in words]

    cleaned_text = ' '.join(cleaned_words)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    return cleaned_text

df['cleaned_text'] = df['colluqial_transformed_text'].apply(clean_text)

df = df[df["cleaned_text"] != ""].reset_index(drop=True)
df

,Date,Username,text,colluqial_transformed_text,cleaned_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,"enggak nely___ haha, poin bag...",enggak nely__ haha poin bagus benar bayi baru ...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,min kalo mau pindah dari bpjs peserta mandiri...,min kalo mau pindah dari bpjs peserta mandiri ...
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...,bpjs pbi akhir bulan kemarin serentak pada non...
...,...,...,...,...,...
574,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,"bpjs pbi bagaimana ini ??? ini lho, ada rak...",bpjs pbi bagaimana ini ini lho ada rakyat yang...
575,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,bpjs pbi yang diblokir wowo. sampai bingung ...,bpjs pbi yang diblokir wowo sampai bingung sal...
576,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,malamm he don't understand about mbak lansia...,malamm he don t understand about mbak lansia l...
577,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga...","kemensos kerja nyata, semoga dengan adanya pe...",kemensos kerja nyata semoga dengan adanya pemu...


In [ ]:
def translate_text(text):
    if not text or not isinstance(text, str) or text.strip() == "":
        return text

    try:
        translated = ts.translate_text(
            text,
            translator='bing',
            from_language="auto",
            to_language="id",
            timeout=10
        )

        return translated if translated else text
    except Exception:
        return text

df['translated_text'] = df["cleaned_text"].apply(translate_text)
df

,Date,Username,text,colluqial_transformed_text,cleaned_text,translated_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...,"Waktu itu aku juga kasusnya sama seperti kamu,..."
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,"enggak nely___ haha, poin bag...",enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,min kalo mau pindah dari bpjs peserta mandiri...,min kalo mau pindah dari bpjs peserta mandiri ...,"Min, kalau mau pindah dari BPJS peserta mandir..."
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...,bpjs pbi akhir bulan kemarin serentak pada non...,BPJS PBI akhir bulan kemarin serentak dinonakt...
...,...,...,...,...,...,...
574,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,"bpjs pbi bagaimana ini ??? ini lho, ada rak...",bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...
575,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,bpjs pbi yang diblokir wowo. sampai bingung ...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...
576,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,malamm he don't understand about mbak lansia...,malamm he don t understand about mbak lansia l...,"malam, dia tidak mengerti tentang mbak lansia ..."
577,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga...","kemensos kerja nyata, semoga dengan adanya pe...",kemensos kerja nyata semoga dengan adanya pemu...,"Kemensos kerja nyata, semoga dengan adanya pem..."


In [ ]:
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

corpus_file = "/kaggle/input/wiki-indonesian-corpus/wiki.txt"
sym_spell.create_dictionary(corpus_file)

def correct_text(text):
    words = text.split()
    corrected_words = []

    for word in words:
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        corrected_words.append(suggestions[0].term if suggestions else word)

    return ' '.join(corrected_words)

df['corrected_text'] = df["translated_text"].apply(correct_text)
df

2026-03-16 16:45:52,365: E symspellpy.symspellpy] Corpus not found at /kaggle/input/wiki-indonesian-corpus/wiki.txt.
ERROR:symspellpy.symspellpy:Corpus not found at /kaggle/input/wiki-indonesian-corpus/wiki.txt.


,Date,Username,text,colluqial_transformed_text,cleaned_text,translated_text,corrected_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...,"Waktu itu aku juga kasusnya sama seperti kamu,...","Waktu itu aku juga kasusnya sama seperti kamu,..."
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,"enggak nely___ haha, poin bag...",enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,min kalo mau pindah dari bpjs peserta mandiri...,min kalo mau pindah dari bpjs peserta mandiri ...,"Min, kalau mau pindah dari BPJS peserta mandir...","Min, kalau mau pindah dari BPJS peserta mandir..."
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...,bpjs pbi akhir bulan kemarin serentak pada non...,BPJS PBI akhir bulan kemarin serentak dinonakt...,BPJS PBI akhir bulan kemarin serentak dinonakt...
...,...,...,...,...,...,...,...
574,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,"bpjs pbi bagaimana ini ??? ini lho, ada rak...",bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...
575,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,bpjs pbi yang diblokir wowo. sampai bingung ...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...
576,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,malamm he don't understand about mbak lansia...,malamm he don t understand about mbak lansia l...,"malam, dia tidak mengerti tentang mbak lansia ...","malam, dia tidak mengerti tentang mbak lansia ..."
577,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga...","kemensos kerja nyata, semoga dengan adanya pe...",kemensos kerja nyata semoga dengan adanya pemu...,"Kemensos kerja nyata, semoga dengan adanya pem...","Kemensos kerja nyata, semoga dengan adanya pem..."


In [ ]:
df['corrected_text'] = df['corrected_text'].str.lower()
df

,Date,Username,text,colluqial_transformed_text,cleaned_text,translated_text,corrected_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...,"Waktu itu aku juga kasusnya sama seperti kamu,...","waktu itu aku juga kasusnya sama seperti kamu,..."
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,"enggak nely___ haha, poin bag...",enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,min kalo mau pindah dari bpjs peserta mandiri...,min kalo mau pindah dari bpjs peserta mandiri ...,"Min, kalau mau pindah dari BPJS peserta mandir...","min, kalau mau pindah dari bpjs peserta mandir..."
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...,bpjs pbi akhir bulan kemarin serentak pada non...,BPJS PBI akhir bulan kemarin serentak dinonakt...,bpjs pbi akhir bulan kemarin serentak dinonakt...
...,...,...,...,...,...,...,...
574,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,"bpjs pbi bagaimana ini ??? ini lho, ada rak...",bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...
575,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,bpjs pbi yang diblokir wowo. sampai bingung ...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...
576,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,malamm he don't understand about mbak lansia...,malamm he don t understand about mbak lansia l...,"malam, dia tidak mengerti tentang mbak lansia ...","malam, dia tidak mengerti tentang mbak lansia ..."
577,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga...","kemensos kerja nyata, semoga dengan adanya pe...",kemensos kerja nyata semoga dengan adanya pemu...,"Kemensos kerja nyata, semoga dengan adanya pem...","kemensos kerja nyata, semoga dengan adanya pem..."


In [ ]:
import nltk
nltk.download('punkt_tab')
def tokenize_text(text):
    return word_tokenize(text)

df['tokenized_text'] = df['corrected_text'].apply(tokenize_text)
df

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,Date,Username,text,colluqial_transformed_text,cleaned_text,translated_text,corrected_text,tokenized_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,"[work, sender, enggak, nyaman, sama, lingkunga..."
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...,"Waktu itu aku juga kasusnya sama seperti kamu,...","waktu itu aku juga kasusnya sama seperti kamu,...","[waktu, itu, aku, juga, kasusnya, sama, sepert..."
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,"enggak nely___ haha, poin bag...",enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...,"[enggak, nely__, haha, poin, bagus, benar, bay..."
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,min kalo mau pindah dari bpjs peserta mandiri...,min kalo mau pindah dari bpjs peserta mandiri ...,"Min, kalau mau pindah dari BPJS peserta mandir...","min, kalau mau pindah dari bpjs peserta mandir...","[min, ,, kalau, mau, pindah, dari, bpjs, peser..."
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...,bpjs pbi akhir bulan kemarin serentak pada non...,BPJS PBI akhir bulan kemarin serentak dinonakt...,bpjs pbi akhir bulan kemarin serentak dinonakt...,"[bpjs, pbi, akhir, bulan, kemarin, serentak, d..."
...,...,...,...,...,...,...,...,...
574,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,"bpjs pbi bagaimana ini ??? ini lho, ada rak...",bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...,"[bpjs, pbi, bagaimana, ini, ini, lho, ada, rak..."
575,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,bpjs pbi yang diblokir wowo. sampai bingung ...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...,"[bpjs, pbi, yang, diblokir, wowo, sampai, bing..."
576,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,malamm he don't understand about mbak lansia...,malamm he don t understand about mbak lansia l...,"malam, dia tidak mengerti tentang mbak lansia ...","malam, dia tidak mengerti tentang mbak lansia ...","[malam, ,, dia, tidak, mengerti, tentang, mbak..."
577,3/2/2026,maysa99628,"@kemensosri kemensos kerja nyata, semoga denga...","kemensos kerja nyata, semoga dengan adanya pe...",kemensos kerja nyata semoga dengan adanya pemu...,"Kemensos kerja nyata, semoga dengan adanya pem...","kemensos kerja nyata, semoga dengan adanya pem...","[kemensos, kerja, nyata, ,, semoga, dengan, ad..."


In [ ]:
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())

def remove_stopwords(tokens):
    if tokens is None:
        return None  # Kembalikan None untuk data yang sudah None
    try:
        filtered_tokens = [token for token in tokens if token.lower() not in stopwords]
        return filtered_tokens if filtered_tokens else None  # Kembalikan None jika hasil kosong
    except:
        return None  # Kembalikan None jika terjadi error

# Terapkan penghapusan stopwords
df['filtered_tokens'] = df['tokenized_text'].apply(remove_stopwords)

# Hapus baris dengan nilai NA di kolom filtered_tokens
df = df.dropna(subset=['filtered_tokens']).reset_index(drop=True)

# Gabungkan token menjadi teks
df['filtered_text'] = df['filtered_tokens'].apply(lambda tokens: " ".join(tokens))

In [ ]:
# Stemming menggunakan Sastrawi
stemmer = StemmerFactory().create_stemmer()

def stemm_word(text):
    return stemmer.stem(text)

df['stemmed_text'] = df["filtered_text"].apply(stemm_word)
df

,Date,Username,text,colluqial_transformed_text,cleaned_text,translated_text,corrected_text,tokenized_text,filtered_tokens,filtered_text,stemmed_text
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,"[work, sender, enggak, nyaman, sama, lingkunga...","[work, sender, enggak, nyaman, sama, lingkunga...",work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkung kerja ...
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...,"Waktu itu aku juga kasusnya sama seperti kamu,...","waktu itu aku juga kasusnya sama seperti kamu,...","[waktu, itu, aku, juga, kasusnya, sama, sepert...","[waktu, aku, kasusnya, sama, kamu, ,, aku, pbi...","waktu aku kasusnya sama kamu , aku pbi disetuj...",waktu aku kasus sama kamu aku pbi tuju siap cu...
2,12/2/2025,grok,@antonsubagia @bank_joee_ @susno2g @jediimar @...,"enggak nely___ haha, poin bag...",enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...,enggak nely__ haha poin bagus benar bayi baru ...,"[enggak, nely__, haha, poin, bagus, benar, bay...","[enggak, nely__, haha, poin, bagus, benar, bay...",enggak nely__ haha poin bagus benar bayi baru ...,enggak nely haha poin bagus benar bayi baru la...
3,12/3/2025,yourrsse,@bpjskesehatanri min kalo mau pindah dari bpjs...,min kalo mau pindah dari bpjs peserta mandiri...,min kalo mau pindah dari bpjs peserta mandiri ...,"Min, kalau mau pindah dari BPJS peserta mandir...","min, kalau mau pindah dari bpjs peserta mandir...","[min, ,, kalau, mau, pindah, dari, bpjs, peser...","[min, ,, kalau, mau, pindah, bpjs, peserta, ma...","min , kalau mau pindah bpjs peserta mandiri bp...",min kalau mau pindah bpjs serta mandiri bpjs p...
4,12/3/2025,hamiyaaaa,bpjs pbi akhir bln kemarin serentak pada nonak...,bpjs pbi akhir bulan kemarin serentak pada non...,bpjs pbi akhir bulan kemarin serentak pada non...,BPJS PBI akhir bulan kemarin serentak dinonakt...,bpjs pbi akhir bulan kemarin serentak dinonakt...,"[bpjs, pbi, akhir, bulan, kemarin, serentak, d...","[bpjs, pbi, akhir, bulan, kemarin, serentak, d...",bpjs pbi akhir bulan kemarin serentak dinonakt...,bpjs pbi akhir bulan kemarin serentak nonaktif
...,...,...,...,...,...,...,...,...,...,...,...
574,3/2/2026,subisaja,@tanyarlfes bpjs @bpjskesehatanri pbi gimana i...,"bpjs pbi bagaimana ini ??? ini lho, ada rak...",bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...,bpjs pbi bagaimana ini ini lho ada rakyat yang...,"[bpjs, pbi, bagaimana, ini, ini, lho, ada, rak...","[bpjs, pbi, bagaimana, lho, rakyat, butuh, per...",bpjs pbi bagaimana lho rakyat butuh perhatian ...,bpjs pbi bagaimana lho rakyat butuh perhati be...
575,3/2/2026,dariguruhonorer,@heywannaknoww @nyunssi bpjs pbi yang diblokir...,bpjs pbi yang diblokir wowo. sampai bingung ...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...,bpjs pbi yang diblokir wowo sampai bingung sal...,"[bpjs, pbi, yang, diblokir, wowo, sampai, bing...","[bpjs, pbi, diblokir, wowo, bingung, salah, gu...",bpjs pbi diblokir wowo bingung salah gue mana ...,bpjs pbi blokir wowo bingung salah gue mana ti...
576,3/2/2026,perju_angan,@jam1malamm @realdonaldtrump he don't understa...,malamm he don't understand about mbak lansia...,malamm he don t understand about mbak lansia l...,"malam, dia tidak mengerti tentang mbak lansia ...","malam, dia tidak mengerti tentang mbak lansia ...","[malam, ,, dia, tidak, mengerti, tentang, mbak...","[malam, ,, mengerti, mbak, lansia, lebih, butu...","malam , mengerti mbak lansia lebih butuh bpjs ...",malam erti mbak lansia lebih butuh bpjs pbi s

# Pelabelan Emosi

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt to NRC-emotion-lexicon-wordlevel-alphabetized-v0.92 (2).txt


In [ ]:
def load_emolex_dictionary(file_path):

    emolex_df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["word","emotion","association"]
    )

    emolex_dict = {}

    for _, row in emolex_df.iterrows():
        if row["association"] == 1:

            word = row["word"]
            emotion = row["emotion"]

            if word not in emolex_dict:
                emolex_dict[word] = []

            emolex_dict[word].append(emotion)

    return emolex_dict


emolex_dict = load_emolex_dictionary(
"/content/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92 (1).txt"
)

In [ ]:
def label_text_with_emolex(
    text,
    emolex_dict,
    use_threshold=False,
    threshold=0.15,
    negation_words=None,
    negation_window=2
):
    # Default daftar kata negasi jika tidak diberikan
    if negation_words is None:
        negation_words = ['tidak', 'bukan', 'jangan', 'belum', 'tanpa', 'kurang']
    # Inisialisasi struktur hasil
    result = {
        'text': text,
        'dominant_emotion': None,
        'dominant_sentiment': None,
        'all_emotions': [],  # Semua emosi yang terdeteksi
        'emotion_scores': {  # Skor mentah untuk tiap emosi
            'anger': 0,
            'anticipation': 0,
            'disgust': 0,
            'fear': 0,
            'joy': 0,
            'sadness': 0,
            'surprise': 0,
            'trust': 0
        },
        'sentiment_scores': {  # Skor mentah untuk sentimen
            'positive': 0,
            'negative': 0
        }
    }
    # Inisialisasi penghitung untuk emosi dan sentimen
    emotion_counter = Counter()
    sentiment_counter = Counter()
    # Tokenisasi teks ke dalam kata-kata (huruf dan angka)
    words = re.findall(r'\b\w+\b', str(text).lower())
    total_words = len(words)
    # Iterasi setiap kata dalam teks
    for i, word in enumerate(words):
        is_negated = False
        # Cek apakah kata berada dalam jangkauan kata negasi
        for j in range(max(0, i - negation_window), i):
            if words[j] in negation_words:
                is_negated = True
                break
        # Jika kata ada dalam kamus EmoLex
        if word in emolex_dict:
            # Tambahkan emosi yang terkait
            for emotion in emolex_dict[word]['emotions']:
                emotion_counter[emotion] += 1
                result['emotion_scores'][emotion] += 1
                # Hindari duplikasi dalam all_emotions
                if emotion not in result['all_emotions']:
                    result['all_emotions'].append(emotion)
            # Tambahkan sentimen dengan mempertimbangkan negasi
            for sentiment in emolex_dict[word]['sentiment']:
                if is_negated:
                    # Balik sentimen jika terdeteksi negasi
                    flipped = 'negative' if sentiment == 'positive' else 'positive'
                    sentiment_counter[flipped] += 1
                    result['sentiment_scores'][flipped] += 1
                else:
                    sentiment_counter[sentiment] += 1
                    result['sentiment_scores'][sentiment] += 1
    # Normalisasi skor emosi berdasarkan jumlah kata
    for emotion in result['emotion_scores']:
        result['emotion_scores'][emotion] /= total_words if total_words > 0 else 1
    # Normalisasi skor sentimen juga
    for sentiment in result['sentiment_scores']:
        result['sentiment_scores'][sentiment] /= total_words if total_words > 0 else 1
    # Menentukan emosi dominan
    if emotion_counter:
        top_emotion, top_emotion_count = emotion_counter.most_common(1)[0]
        if use_threshold:
            if total_words > 0 and (top_emotion_count / total_words) >= threshold:
                result['dominant_emotion'] = top_emotion
        else:
            result['dominant_emotion'] = top_emotion
    # Menentukan sentimen dominan
    if sentiment_counter:
        pos = result['sentiment_scores']['positive']
        neg = result['sentiment_scores']['negative']
        diff = abs(pos - neg)
        if use_threshold:
            # Jika selisih kecil, anggap netral
            if diff < threshold:
                result['dominant_sentiment'] = 'neutral'
            else:
                result['dominant_sentiment'] = 'positive' if pos > neg else 'negative'
        else:
            # Jika selisih sangat kecil (<0.05), anggap netral
            result['dominant_sentiment'] = (
                'neutral' if diff < 0.05 else
                ('positive' if pos > neg else 'negative')
            )
    return result

In [ ]:
import re
from collections import Counter

def label_text_with_emolex(
    text,
    emolex_dict, # This now contains word: [emotion1, emotion2, ...]
    use_threshold=False,
    threshold=0.15,
    negation_words=None,
    negation_window=2
):
    if negation_words is None:
        negation_words = ['tidak', 'bukan', 'jangan', 'belum', 'tanpa', 'kurang']

    result = {
        'text': text,
        'dominant_emotion': None,
        'dominant_sentiment': None,
        'all_emotions': [],
        'emotion_scores': {
            'anger': 0, 'anticipation': 0, 'disgust': 0, 'fear': 0,
            'joy': 0, 'sadness': 0, 'surprise': 0, 'trust': 0
        },
        'sentiment_scores': {
            'positive': 0, 'negative': 0
        }
    }

    emotion_counter = Counter() # For specific emotions
    sentiment_counter = Counter() # For positive/negative sentiment

    words = re.findall(r'\b\w+\b', str(text).lower())
    total_words = len(words)

    for i, word in enumerate(words):
        is_negated = False
        for j in range(max(0, i - negation_window), i):
            if words[j] in negation_words:
                is_negated = True
                break

        if word in emolex_dict:
            for category in emolex_dict[word]: # category can be an emotion or a sentiment label
                if category == 'positive':
                    sentiment_to_add = 'positive' if not is_negated else 'negative'
                    sentiment_counter[sentiment_to_add] += 1
                    result['sentiment_scores'][sentiment_to_add] += 1
                elif category == 'negative':
                    sentiment_to_add = 'negative' if not is_negated else 'positive'
                    sentiment_counter[sentiment_to_add] += 1
                    result['sentiment_scores'][sentiment_to_add] += 1
                else: # It's one of the 8 specific emotions
                    emotion = category
                    if emotion in result['emotion_scores']: # Only count if it's one of the predefined emotions
                        emotion_counter[emotion] += 1
                        result['emotion_scores'][emotion] += 1
                        if emotion not in result['all_emotions']:
                            result['all_emotions'].append(emotion)

    # Normalize scores
    for emotion in result['emotion_scores']:
        result['emotion_scores'][emotion] /= total_words if total_words > 0 else 1
    for sentiment in result['sentiment_scores']:
        result['sentiment_scores'][sentiment] /= total_words if total_words > 0 else 1

    # Determine dominant emotion
    if emotion_counter:
        top_emotion, top_emotion_count = emotion_counter.most_common(1)[0]
        if use_threshold:
            if total_words > 0 and (top_emotion_count / total_words) >= threshold:
                result['dominant_emotion'] = top_emotion
        else:
            result['dominant_emotion'] = top_emotion

    # Determine dominant sentiment
    if sentiment_counter:
        pos = result['sentiment_scores']['positive']
        neg = result['sentiment_scores']['negative']
        diff = abs(pos - neg)
        if use_threshold:
            if diff < threshold:
                result['dominant_sentiment'] = 'neutral'
            else:
                result['dominant_sentiment'] = 'positive' if pos > neg else 'negative'
        else:
            result['dominant_sentiment'] = (
                'neutral' if diff < 0.05 else
                ('positive' if pos > neg else 'negative')
            )
    return result

# Fungsi untuk memberi label emosi dan sentimen ke seluruh DataFrame
def label_dataframe_with_emolex(df, text_column, emolex_dict, use_threshold=False, threshold=0.15):
    emotions = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
    sentiments = ['positive', 'negative']

    for emotion in emotions:
        df[f'emotion_score_{emotion}'] = 0.0
    for sentiment in sentiments:
        df[f'sentiment_score_{sentiment}'] = 0.0

    for col in ['emotion', 'sentiment', 'all_emotions']:
        df[col] = None

    for idx, row in df.iterrows():
        if pd.isna(row[text_column]) or row[text_column] == '':
            continue
        labels = label_text_with_emolex(row[text_column], emolex_dict, use_threshold, threshold)
        df.at[idx, 'emotion'] = labels['dominant_emotion']
        df.at[idx, 'sentiment'] = labels['dominant_sentiment']
        df.at[idx, 'all_emotions'] = ', '.join(labels['all_emotions']) if labels['all_emotions'] else None

        for emotion in emotions:
            df.at[idx, f'emotion_score_{emotion}'] = labels['emotion_scores'][emotion]
        for sentiment in sentiments:
            df.at[idx, f'sentiment_score_{sentiment}'] = labels['sentiment_scores'][sentiment]

    return df

# Terapkan pelabelan ke DataFrame
df = label_dataframe_with_emolex(df, 'filtered_text', emolex_dict)

In [ ]:
df1 = pd.read_csv("/content/pre-processing.csv")
df1

,Date,Username,text,colluqial_transformed_text,cleaned_text,translated_text,corrected_text,tokenized_text,filtered_tokens,filtered_text,...,emotion_score_fear,emotion_score_joy,emotion_score_sadness,emotion_score_surprise,emotion_score_trust,sentiment_score_positive,sentiment_score_negative,emotion,sentiment,all_emotions
0,12/1/2025,worksfess,work! sender gak nyaman sama lingkungan kerja ...,work! sender enggak nyaman sama lingkungan ker...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,work sender enggak nyaman sama lingkungan kerj...,"['work', 'sender', 'enggak', 'nyaman', 'sama',...","['work', 'sender', 'enggak', 'nyaman', 'sama',...",work sender enggak nyaman sama lingkungan kerj...,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.047619,disgust,neutral,disgust
1,12/2/2025,mollypinki,@undipmfs2 waktu itu aku case-nya juga sama ka...,"waktu itu aku case-nya juga sama kayak kamu, ...",waktu itu aku case nya juga sama kayak kamu ak...,waktu itu aku case nya juga sama kayak kamu ak...,waktu itu aku case nya juga sama kayak kamu ak...,"['waktu', 'itu', 'aku', 'case', 'nya', 'juga',...","['waktu', 'aku', 'case', 'nya', 'sama', 'kayak...",waktu aku case nya sama kayak kamu aku pbi dis...,...,0.045455,0.000000,0.045455,0.000000,0.000000,0.000000,0.045455,fear,neutral,"fear, sadness"
2,12/6/2025,incess_juli99,@lesdeaux maaf kak dibikin pisah aja maybe kk ...,maaf kak dibikin pisah saja maybe kakak nya k...,maaf kak dibikin pisah saja maybe kakak nya ka...,maaf kak dibikin pisah saja maybe kakak nya ka...,maaf kak dibikin pisah saja maybe kakak nya ka...,"['maaf', 'kak', 'dibikin', 'pisah', 'saja', 'm...","['maaf', 'kak', 'dibikin', 'pisah', 'maybe', '...",maaf kak dibikin pisah maybe kakak nya kayak u...,...,0.023810,0.000000,0.000000,0.023810,0.000000,0.000000,0.023810,anticipation,neutral,"anticipation, fear, surprise"
3,12/12/2025,BPJSKesehatanRI,@rumahsajakku salam. perubahan faskes bisa dil...,salam. perubahan faskes bisa dilakukan melalu...,salam perubahan faskes bisa dilakukan melalui ...,salam perubahan faskes bisa dilakukan melalui ...,salam perubahan faskes bisa dilakukan melalui ...,"['salam', 'perubahan', 'faskes', 'bisa', 'dila...","['salam', 'perubahan', 'faskes', 'dilakukan', ...",salam perubahan faskes dilakukan melalui mobil...,...,0.000000,0.031250,0.000000,0.031250,0.000000,0.031250,0.000000,anticipation,neutral,"anticipation, joy, surprise"
4,12/18/2025,pityofme,@jogja_base ternyata udah masuk pbi\n jadi ga ...,ternyata sudah masuk pbi jadi enggak daftar m...,ternyata sudah masuk pbi jadi enggak daftar me...,ternyata sudah masuk pbi jadi enggak daftar me...,ternyata sudah masuk pbi jadi enggak daftar me...,"['ternyata', 'sudah', 'masuk', 'pbi', 'jadi', ...","['ternyata', 'masuk', 'pbi', 'jadi', 'enggak',...",ternyata masuk pbi jadi enggak daftar meskipun...,...,0.027778,0.000000,0.027778,0.000000,0.000000,0.027778,0.027778,anger,neutral,"anger, disgust, fear, sadness"
5,12/20/2025,atsara_nia,@dedeags3006 @piyusaja2 duh polosnya itu otak ...,duh polosnya itu otak 🤣 memang semua itu gra...,duh polosnya itu otak memang semua itu gratis ...,duh polosnya itu otak memang semua itu gratis ...,duh polosnya itu otak memang semua itu gratis ...,"['duh', 'polosnya', 'itu', 'otak', 'memang', '...","['duh', 'polosnya', 'otak', 'memang', 'semua',...",duh polosnya otak memang semua gratis bpjs bay...,...,0.034483,0.000000,0.034483,0.000000,0.000000,0.000000,0.034483,fear,neutral,"fear, sadness"
6,12/25/2025,Dinarra93,"@bpjskesehatanri izin bertanya, bulek sy suami...","izin bertanya, bulek saya suami istri bpjs pb...",izin bertanya bulek saya suami istri bpjs pbi ...,izin bertanya bulek saya suami istri bpjs pbi ...,izin bertanya bulek saya suami istri bpjs pbi ...,"['izin', 'bertanya', 'bulek', 'saya', 'suami',...","['izin', 'bertanya', 'bulek', 'suami', 'istri'...",izin bertanya bulek suami istri bpjs pbi mobil...,...,0.000000,0.000000,0.000000,0.000000,0.030303

# Tahap pengecekan

In [ ]:
df['result'] = df['stemmed_text'].apply(
    lambda x: label_text_with_emolex(x, emolex_dict)
)

df[['stemmed_text','result']]

,stemmed_text,result
0,work sender enggak nyaman sama lingkung kerja ...,{'text': 'work sender enggak nyaman sama lingk...
1,waktu aku kasus sama kamu aku pbi tuju siap cu...,{'text': 'waktu aku kasus sama kamu aku pbi tu...
2,enggak nely haha poin bagus benar bayi baru la...,{'text': 'enggak nely haha poin bagus benar ba...
3,min kalau mau pindah bpjs serta mandiri bpjs p...,{'text': 'min kalau mau pindah bpjs serta mand...
4,bpjs pbi akhir bulan kemarin serentak nonaktif,{'text': 'bpjs pbi akhir bulan kemarin serenta...
...,...,...
574,bpjs pbi bagaimana lho rakyat butuh perhati be...,{'text': 'bpjs pbi bagaimana lho rakyat butuh ...
575,bpjs pbi blokir wowo bingung salah gue mana ti...,{'text': 'bpjs pbi blokir wowo bingung salah g...
576,malam erti mbak lansia lebih butuh bpjs pbi se...,{'text': 'malam erti mbak lansia lebih butuh b...
577,kemensos kerja nyata moga ada mutakhir data bp...,{'text': 'kemensos kerja nyata moga ada mutakh...


In [ ]:
len(df)

579

In [ ]:
df['emotion'].value_counts(dropna=False)

,count
emotion,
None,529
trust,18
anger,14
anticipation,11
fear,3
joy,3
disgust,1
